# 20 v2 - Selective-refinement LIBERO-PRO worker: model 0

This collects the 10-episodes-per-task subset of the 13-suite cohort: 1,300 identities per arm.
It uses the established K=5, Euler-steps `(3,4)`, refine-last configuration and requests baseline and
refinement under the existing `pi05-diversity-signal-v2-m0` experiment. Supabase
rollout IDs are behavior-derived, so the existing 130 baselines and any completed refinements are
reused; only missing rows execute.

The model repository and immutable revision are read from the existing baseline run metadata.
This fails rather than silently evaluating a newer Hub upload. Run `SHARD_INDEX=0,1,2,3`; each
completed shard is resumable. Every fifty new rollouts, the worker prints the current baseline SR,
current refinement SR, and historical reference SR by suite.

## 1. Setup a fresh GPU runtime

In [ ]:
EXTRAS = 'sim'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Configuration and resumable collection

In [ ]:
from pathlib import Path
from google.colab import drive
from pnp.config import PI05_REPO_ID
from pnp.diversity import (DIVERSITY_V2_EXPERIMENT_PREFIX,
    load_bootstrap_manifest, run_diversity_refinement_worker)

drive.mount("/content/drive")

MEMBER_INDEX = 0
EPISODES_PER_TASK = 10         # 13 suites x 10 tasks x 10 episodes = 1,300 identities/arm
SHARD_COUNT = 4
SHARD_INDEX = 0                # run 0, 1, 2, 3 for this member
EXPERIMENT_PREFIX = DIVERSITY_V2_EXPERIMENT_PREFIX
MANIFEST_PATH = Path(
    "/content/drive/MyDrive/pnp_diversity_v2/bootstrap_manifest_finetuned_v2.json")
manifest = load_bootstrap_manifest(MANIFEST_PATH)
assert manifest["source_model"] == PI05_REPO_ID, manifest["source_model"]

print({"member": MEMBER_INDEX, "experiment_prefix": EXPERIMENT_PREFIX,
       "episodes_per_task": EPISODES_PER_TASK,
       "shard_count": SHARD_COUNT, "shard_index": SHARD_INDEX,
       "manifest_hash": manifest["manifest_hash"]})
run_diversity_refinement_worker(
    member_index=MEMBER_INDEX,
    episodes_per_task=EPISODES_PER_TASK,
    shard_count=SHARD_COUNT, shard_index=SHARD_INDEX,
    manifest_hash=manifest["manifest_hash"],
    experiment_prefix=EXPERIMENT_PREFIX)